In [1]:
import zipfile
from pathlib import Path

MOT17_ZIP = Path('../data/MOT17.zip')
MOT20_ZIP = Path('../data/MOT20.zip')

for zip_path in [MOT17_ZIP, MOT20_ZIP]:
    print(f'=== {zip_path.name} ===')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = zf.namelist()
        # Show the first few paths to confirm the internal folder structure
        for n in names[:5]:
            print(' ', n)
        # List every unique top-level train sequence found
        seqs = sorted(set(
            n.split('/')[2] for n in names
            if '/train/' in n and len(n.split('/')) > 3
        ))
        print(f'  Sequences found: {seqs}')
    print()

=== MOT17.zip ===
  MOT17/
  MOT17/test/
  MOT17/test/MOT17-14-SDP/
  MOT17/test/MOT17-14-SDP/det/
  MOT17/test/MOT17-14-SDP/det/det.txt
  Sequences found: ['MOT17-02-DPM', 'MOT17-02-FRCNN', 'MOT17-02-SDP', 'MOT17-04-DPM', 'MOT17-04-FRCNN', 'MOT17-04-SDP', 'MOT17-05-DPM', 'MOT17-05-FRCNN', 'MOT17-05-SDP', 'MOT17-09-DPM', 'MOT17-09-FRCNN', 'MOT17-09-SDP', 'MOT17-10-DPM', 'MOT17-10-FRCNN', 'MOT17-10-SDP', 'MOT17-11-DPM', 'MOT17-11-FRCNN', 'MOT17-11-SDP', 'MOT17-13-DPM', 'MOT17-13-FRCNN', 'MOT17-13-SDP']

=== MOT20.zip ===
  MOT20/
  MOT20/train/
  MOT20/train/MOT20-02/
  MOT20/train/MOT20-02/gt/
  MOT20/train/MOT20-02/gt/gt.txt
  Sequences found: ['MOT20-01', 'MOT20-02', 'MOT20-03', 'MOT20-05']



In [2]:
NEEDED_MOT17 = ['MOT17-02-FRCNN', 'MOT17-04-FRCNN']

with zipfile.ZipFile(MOT17_ZIP, 'r') as zf:
    targets = [
        n for n in zf.namelist()
        if any(f'train/{seq}/img1/' in n for seq in NEEDED_MOT17)
    ]
    print(f'Extracting {len(targets)} MOT17 image files...')
    for n in targets:
        zf.extract(n, Path('../data'))

print('MOT17 image extraction complete.')

Extracting 1652 MOT17 image files...
MOT17 image extraction complete.


In [3]:
with zipfile.ZipFile(MOT20_ZIP, 'r') as zf:
    seqs = sorted(set(
        n.split('/')[2] for n in zf.namelist()
        if '/train/' in n and len(n.split('/')) > 3
    ))
print(f'MOT20 training sequences available: {seqs}')

MOT20 training sequences available: ['MOT20-01', 'MOT20-02', 'MOT20-03', 'MOT20-05']


In [4]:
import cv2
import configparser
from pathlib import Path

def stitch_full_resolution_video(seq_name: str, base_dir: str = 'MOT17',
                                   output_dir: Path = Path('../data/MOT17/videos')):
    seq_dir = Path(f'../data/{base_dir}/train/{seq_name}')
    img_dir = seq_dir / 'img1'
    ini_path = seq_dir / 'seqinfo.ini'

    assert img_dir.exists(), f'No raw frames found: {img_dir}'
    assert ini_path.exists(), f'No seqinfo.ini found: {ini_path}'

    config = configparser.ConfigParser()
    config.read(ini_path)
    info = config['Sequence']
    fps, width, height = float(info['frameRate']), int(info['imWidth']), int(info['imHeight'])
    seq_length, im_ext = int(info['seqLength']), info['imExt']

    print(f'{seq_name}: {width}x{height} @ {fps}fps, {seq_length} frames')
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f'{seq_name}-fullres.mp4'

    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    for i in range(1, seq_length + 1):
        frame = cv2.imread(str(img_dir / f'{i:06d}{im_ext}'))
        assert frame is not None, f'Could not read frame {i}'
        writer.write(frame)
    writer.release()
    print(f'Wrote: {out_path}')

stitch_full_resolution_video('MOT17-02-FRCNN')
stitch_full_resolution_video('MOT17-04-FRCNN')

MOT17-02-FRCNN: 1920x1080 @ 30.0fps, 600 frames
Wrote: ..\data\MOT17\videos\MOT17-02-FRCNN-fullres.mp4
MOT17-04-FRCNN: 1920x1080 @ 30.0fps, 1050 frames
Wrote: ..\data\MOT17\videos\MOT17-04-FRCNN-fullres.mp4
